In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_INPUTS = True
REUSE_PROPOSALS = False
REUSE_REACQUISITION = False
REUSE_BRIDGE = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Secondary Distal Compact-Lumen Reacquisition

Starts from the last unequivocally compact secondary-branch segment (~12.2 mm). It independently searches 1.2–5.5 mm forward for reappearing coronary-sized lumen tracklets. No connected lumen through the ambiguous zone is required. Vesselness is proposal-only; serial source-CCTA compact-lumen validation determines reacquisition. Only then is a short bridge assessed.


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -q --depth 1 --branch secondary-distal-reacquisition-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.secondary_distal_reacquisition import (
    SecondaryDistalReacquisitionWorkflow,
    synthetic_reacquisition_self_test,
)

self_test = synthetic_reacquisition_self_test()
display(self_test)
assert self_test['passed'], self_test

wf = SecondaryDistalReacquisitionWorkflow(
    root='/content/drive/MyDrive/OpenPlaque',
    reuse={
        'inputs': REUSE_INPUTS,
        'proposals': REUSE_PROPOSALS,
        'reacquisition': REUSE_REACQUISITION,
        'bridge': REUSE_BRIDGE,
        'figures': REUSE_FIGURES,
        'report': REUSE_REPORT,
    },
)
display(wf.cache_status())


In [ ]:
snapshot = wf.load_inputs(reacquisition_origin_arc_mm=12.20)
display(snapshot)
display(wf.calibration)
print('Reacquisition origin z,y,x:', wf.reacquisition_origin)
print('Origin arc:', wf.reacquisition_origin_arc, 'mm')


In [ ]:
proposals = wf._proposal_candidates(max_points=80, nms_mm=0.70)
print('Independent distal proposals:', len(proposals))
display(proposals.head(20))


In [ ]:
tracks = wf.search_reacquisition(top_proposals=60)
print('Tracklet candidates tested:', len(tracks))
print('Supported compact-lumen tracklets:', int(tracks.tracklet_gate.sum()) if len(tracks) else 0)
display(tracks.head(20))
if len(wf.best_tracklet_qc):
    display(wf.best_tracklet_qc)


In [ ]:
bridge = wf.assess_bridge()
print('Bridge candidates:', len(bridge))
display(bridge)


In [ ]:
summary = wf.run()
display(summary)


In [ ]:
names = wf.make_figures()
for name in names:
    path = str(wf.out / name)
    print(name)
    display(Image(filename=path))


In [ ]:
zip_path = wf.package()
print('Final ZIP:', zip_path)
print('Report:', wf.out / 'OPENPLAQUE_SECONDARY_DISTAL_REACQUISITION_REPORT.html')
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_SECONDARY_DISTAL_REACQUISITION_REPORT_BACK.zip')
